# Zgodność anotatorów – iteracja 1

Anotatorzy: **Igor** i **Kacper**. Porównujemy tylko teksty zaanotowane przez obu.

- poziom całego tekstu: `stance` (popiera / neutralna / sprzeciwia się), `ton` (zwykły / agresywny)
- poziom fragmentów: spany `Agresja`, `Kontekst` – zgodność liczona na tokenach (etykieta tokenu: `O` / `Agresja` / `Kontekst`) oraz F1 na spanach

In [1]:
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

ANNOTATORS = {
    "igor": "annotations/igor-annotations.json",
    "kacper": "annotations/kacper-annotations.json",
}
data = {name: {t["id"]: t for t in json.load(open(path))} for name, path in ANNOTATORS.items()}
common = sorted(set.intersection(*(set(d) for d in data.values())))
for name, d in data.items():
    print(f"{name}: {len(d)} tekstów, brak u drugiego: {sorted(set(d) - set(common))}")
print("wspólne:", len(common))

igor: 50 tekstów, brak u drugiego: [6]
kacper: 49 tekstów, brak u drugiego: []
wspólne: 49


## Metryki

In [2]:
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    cats = np.union1d(a, b)
    p_o = np.mean(a == b)
    p_e = sum(np.mean(a == c) * np.mean(b == c) for c in cats)
    return (p_o - p_e) / (1 - p_e)


def fleiss_kappa(*raters):
    # macierz n_items x n_categories: ilu anotatorów przypisało daną kategorię
    labels = np.column_stack(raters)
    cats = np.unique(labels)
    counts = np.stack([(labels == c).sum(axis=1) for c in cats], axis=1)
    n = labels.shape[1]
    p_j = counts.sum(axis=0) / counts.sum()
    P_i = ((counts**2).sum(axis=1) - n) / (n * (n - 1))
    P_bar, P_e = P_i.mean(), (p_j**2).sum()
    return (P_bar - P_e) / (1 - P_e)


def interpret(k):
    # skala Landisa i Kocha
    for bound, name in [(0, "brak"), (0.2, "słaba"), (0.4, "przeciętna"), (0.6, "umiarkowana"), (0.8, "znaczna")]:
        if k <= bound:
            return name
    return "prawie pełna"

## Poziom całego tekstu

In [3]:
names = list(data)
doc = pd.DataFrame(
    {f"{field}_{name}": [data[name][i][field] for i in common] for field in ("stance", "ton") for name in names},
    index=common,
)

rows = []
for field in ("stance", "ton"):
    a, b = (doc[f"{field}_{n}"] for n in names)
    ck, fk = cohen_kappa(a, b), fleiss_kappa(a, b)
    rows.append({"pole": field, "zgodność %": 100 * np.mean(a == b), "Cohen κ": ck, "Fleiss κ": fk, "interpretacja": interpret(ck)})
doc_scores = pd.DataFrame(rows).set_index("pole")
doc_scores.round(3)

,zgodność %,Cohen κ,Fleiss κ,interpretacja
pole,,,,
stance,71.429,0.572,0.570,umiarkowana
ton,87.755,0.652,0.648,znaczna


In [4]:
for field in ("stance", "ton"):
    display(pd.crosstab(doc[f"{field}_{names[0]}"], doc[f"{field}_{names[1]}"], rownames=[names[0]], colnames=[names[1]]))

kacper,neutralna,popiera,sprzeciwia się
igor,,,
neutralna,10,1,3
popiera,5,11,1
sprzeciwia się,2,2,14


kacper,agresywny,zwykły
igor,,
agresywny,8,5
zwykły,1,35


## Poziom fragmentów (spany)

In [5]:
TOKEN = re.compile(r"\w+|[^\w\s]")


def token_labels(task):
    labels = []
    for m in TOKEN.finditer(task["text"]):
        label = "O"
        for s in task.get("spans", []):
            if m.start() < s["end"] and s["start"] < m.end():
                label = s["labels"][0]
                break
        labels.append(label)
    return labels


tok = {n: [l for i in common for l in token_labels(data[n][i])] for n in names}
a, b = tok[names[0]], tok[names[1]]
print("tokenów:", len(a))
print({n: Counter(t) for n, t in tok.items()})

rows = [{"etykieta": "wszystkie (O/Agresja/Kontekst)", "Cohen κ": cohen_kappa(a, b), "Fleiss κ": fleiss_kappa(a, b)}]
for label in ("Agresja", "Kontekst"):
    ba, bb = [x == label for x in a], [x == label for x in b]
    rows.append({"etykieta": label, "Cohen κ": cohen_kappa(ba, bb), "Fleiss κ": fleiss_kappa(ba, bb)})
tok_scores = pd.DataFrame(rows).set_index("etykieta")
tok_scores["interpretacja"] = tok_scores["Cohen κ"].map(interpret)
tok_scores.round(3)

tokenów: 3243
{'igor': Counter({'O': 2655, 'Kontekst': 391, 'Agresja': 197}), 'kacper': Counter({'O': 2899, 'Kontekst': 181, 'Agresja': 163})}


,Cohen κ,Fleiss κ,interpretacja
etykieta,,,
wszystkie (O/Agresja/Kontekst),0.260,0.253,przeciętna
Agresja,0.295,0.294,przeciętna
Kontekst,0.209,0.199,przeciętna


Kappa na tokenach jest zawyżana przez dużą liczbę tokenów `O`, więc dodatkowo F1 na spanach
(jeden anotator traktowany jako „złoty standard”, F1 jest symetryczne): dopasowanie dokładne oraz z nakładaniem się.

In [6]:
def spans(name):
    return [(i, s["start"], s["end"], s["labels"][0]) for i in common for s in data[name][i].get("spans", [])]


def span_f1(pred, gold, match):
    tp_p = sum(any(match(p, g) for g in gold) for p in pred)
    tp_g = sum(any(match(p, g) for p in pred) for g in gold)
    prec, rec = tp_p / len(pred), tp_g / len(gold)
    return 2 * prec * rec / (prec + rec) if prec + rec else 0.0


exact = lambda p, g: p == g
overlap = lambda p, g: p[0] == g[0] and p[3] == g[3] and p[1] < g[2] and g[1] < p[2]

rows = []
for label in ("Agresja", "Kontekst", None):
    sa = [s for s in spans(names[0]) if label in (None, s[3])]
    sb = [s for s in spans(names[1]) if label in (None, s[3])]
    rows.append({
        "etykieta": label or "wszystkie",
        f"spany {names[0]}": len(sa),
        f"spany {names[1]}": len(sb),
        "F1 dokładne": span_f1(sa, sb, exact),
        "F1 nakładanie": span_f1(sa, sb, overlap),
    })
span_scores = pd.DataFrame(rows).set_index("etykieta")
span_scores.round(3)

,spany igor,spany kacper,F1 dokładne,F1 nakładanie
etykieta,,,,
Agresja,37,50,0.138,0.368
Kontekst,63,36,0.101,0.404
wszystkie,100,86,0.118,0.387


## Podsumowanie

In [7]:
summary = pd.concat([
    doc_scores[["Cohen κ", "Fleiss κ"]],
    tok_scores[["Cohen κ", "Fleiss κ"]].rename(index=lambda x: f"spany: {x}"),
])
summary["interpretacja"] = summary["Cohen κ"].map(interpret)
summary.round(3)

,Cohen κ,Fleiss κ,interpretacja
stance,0.572,0.570,umiarkowana
ton,0.652,0.648,znaczna
spany: wszystkie (O/Agresja/Kontekst),0.260,0.253,przeciętna
spany: Agresja,0.295,0.294,przeciętna
spany: Kontekst,0.209,0.199,przeciętna


## Teksty z rozbieżnościami

In [8]:
diff = doc[(doc[f"stance_{names[0]}"] != doc[f"stance_{names[1]}"]) | (doc[f"ton_{names[0]}"] != doc[f"ton_{names[1]}"])]
diff = diff.assign(text=[data[names[0]][i]["text"][:150] for i in diff.index])
pd.set_option("display.max_colwidth", 150)
diff

,stance_igor,stance_kacper,ton_igor,ton_kacper,text
5,neutralna,sprzeciwia się,agresywny,zwykły,Też były i też na terenie II Rzeczypospolitej mordowały tysiące Polaków. Czy to są właściwe sformułowania? I jeszcze raz państwu powiem: będziecie...
8,sprzeciwia się,popiera,agresywny,agresywny,"Pan poseł Zandberg pokazuje, jak bardzo realne są zagrożenia, przed którymi ostrzegamy i przed którymi chcemy ratować Polaków. Chcemy ratować Pola..."
11,sprzeciwia się,neutralna,zwykły,zwykły,"W trakcie jego trwania internowano łącznie 10 131 działaczy związanych z Niezależnym Samorządnym Związkiem Zawodowym „Solidarność”, a życie straci..."
13,neutralna,neutralna,agresywny,zwykły,"Dzień dobry. Marszałku! Wysoka Izbo! Panie Pośle Warchoł! Uprzejmie informuję: jeżeli klub nie przygotował panu scenariusza głosowań, to zaraz dow..."
14,popiera,neutralna,zwykły,zwykły,Projekt ten odpowiada na wyzwania związane z dynamicznym rozwojem nowych technologii w transporcie drogowym. Automatyzacja pojazdów to nie jest od...
15,sprzeciwia się,popiera,agresywny,zwykły,"Ustawa pana prezydenta jest bardzo restrykcyjna, bardzo przeregulowuje tę branżę. Powinna wam się podobać, powinniście ją poprzeć. Nie popieracie ..."
16,popiera,neutralna,zwykły,zwykły,"Chcę podziękować, panie marszałku, że pan przywitał polskich patriotów z klubów „Gazety Polskiej”, z Ruchu Obrony Granic, ale, panie marszałku, za..."
23,neutralna,neutralna,agresywny,zwykły,"A więc skierujmy ustawę do pracy na komisji, pracujmy. Zdrada dyplomatyczna się szerzy. Ona została nawet zinstytucjonalizowana. Wyszło przecież w..."
28,popiera,sprzeciwia się,zwykły,zwykły,"Jeśli chodzi o pytania, tu jest absolutne niezrozumienie, wydaje mi się, pomocy, którą dzisiaj omawiamy, dla polskich rolników, dla rolnictwa. To ..."
29,neutralna,sprzeciwia się,zwykły,zwykły,"No i najważniejsza rzecz. Proszę szanownego pana ministra Nenemana, który wszystkie nasze poprawki odrzuca, o odpowiedź na pytanie, czy zgodzi się..."
